In [43]:
# Cart 이탈 분석에 사용할 라이브러리와 퍼널 데이터 경로 설정

import duckdb
from pathlib import Path

first_cart_parquet = r"../data/processed/first_cart_after_view.parquet"
sequential_funnel_parquet = r"../data/processed/session_product_sequential_funnel.parquet"

In [45]:
# 재구성된 분석 세션×상품 기준으로 Cart 이후 구매 완료 / 이탈 분류

cart_abandonment_parquet = r"../data/processed/cart_abandonment.parquet"

Path(cart_abandonment_parquet).unlink(missing_ok=True)

duckdb.sql(f"""
    COPY (
        SELECT
            c.user_id,
            c.user_session,
            c.analysis_session_number,
            c.product_id,
            c.first_view_time,
            c.first_cart_after_view,

            CASE
                WHEN p.first_purchase_after_cart IS NOT NULL
                THEN 1
                ELSE 0
            END AS converted,

            p.first_purchase_after_cart

        FROM read_parquet('{first_cart_parquet}') c

        LEFT JOIN read_parquet('{sequential_funnel_parquet}') p
            ON c.user_id = p.user_id
           AND c.user_session = p.user_session
           AND c.analysis_session_number = p.analysis_session_number
           AND c.product_id = p.product_id

    )
    TO '{cart_abandonment_parquet}'
    (FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [47]:
# 구매 완료와 Cart 이탈 세션×상품 수 확인

duckdb.sql(f"""
    SELECT
        converted,
        COUNT(*) AS session_product_count

    FROM read_parquet('{cart_abandonment_parquet}')

    GROUP BY converted
    ORDER BY converted
""").show()

┌───────────┬───────────────────────┐
│ converted │ session_product_count │
│   int32   │         int64         │
├───────────┼───────────────────────┤
│         0 │               4820961 │
│         1 │               4379438 │
└───────────┴───────────────────────┘



In [49]:
# Cart까지 도달한 세션×상품 중 구매 완료율과 이탈률 계산

duckdb.sql(f"""
    SELECT
        COUNT(*) AS carted_count,

        SUM(CASE WHEN converted = 1
                 THEN 1 ELSE 0 END) AS converted_count,

        SUM(CASE WHEN converted = 0
                 THEN 1 ELSE 0 END) AS abandoned_count,

        ROUND(
            SUM(CASE WHEN converted = 1
                     THEN 1 ELSE 0 END) * 100.0
            / COUNT(*),
            2
        ) AS conversion_rate,

        ROUND(
            SUM(CASE WHEN converted = 0
                     THEN 1 ELSE 0 END) * 100.0
            / COUNT(*),
            2
        ) AS abandonment_rate

    FROM read_parquet('{cart_abandonment_parquet}')
""").show()

┌──────────────┬─────────────────┬─────────────────┬─────────────────┬──────────────────┐
│ carted_count │ converted_count │ abandoned_count │ conversion_rate │ abandonment_rate │
│    int64     │     int128      │     int128      │     double      │      double      │
├──────────────┼─────────────────┼─────────────────┼─────────────────┼──────────────────┤
│      9200399 │         4379438 │         4820961 │            47.6 │             52.4 │
└──────────────┴─────────────────┴─────────────────┴─────────────────┴──────────────────┘



In [51]:
# Cart 발생 월 기준으로 구매 완료율과 Cart 이탈률 계산

duckdb.sql(f"""
    SELECT
        STRFTIME(first_cart_after_view, '%Y-%m') AS month,

        COUNT(*) AS carted_count,

        SUM(
            CASE WHEN converted = 1
            THEN 1 ELSE 0 END
        ) AS converted_count,

        SUM(
            CASE WHEN converted = 0
            THEN 1 ELSE 0 END
        ) AS abandoned_count,

        ROUND(
            SUM(CASE WHEN converted = 1
                     THEN 1 ELSE 0 END) * 100.0
            / COUNT(*),
            2
        ) AS conversion_rate,

        ROUND(
            SUM(CASE WHEN converted = 0
                     THEN 1 ELSE 0 END) * 100.0
            / COUNT(*),
            2
        ) AS abandonment_rate

    FROM read_parquet('{cart_abandonment_parquet}')

    GROUP BY month
    ORDER BY month
""").show()

┌─────────┬──────────────┬─────────────────┬─────────────────┬─────────────────┬──────────────────┐
│  month  │ carted_count │ converted_count │ abandoned_count │ conversion_rate │ abandonment_rate │
│ varchar │    int64     │     int128      │     int128      │     double      │      double      │
├─────────┼──────────────┼─────────────────┼─────────────────┼─────────────────┼──────────────────┤
│ 2019-12 │      2298442 │         1066190 │         1232252 │           46.39 │            53.61 │
│ 2020-01 │      1522267 │          731469 │          790798 │           48.05 │            51.95 │
│ 2020-02 │      1585664 │          782129 │          803535 │           49.33 │            50.67 │
│ 2020-03 │      1861838 │          933979 │          927859 │           50.16 │            49.84 │
│ 2020-04 │      1932188 │          865671 │         1066517 │            44.8 │             55.2 │
└─────────┴──────────────┴─────────────────┴─────────────────┴─────────────────┴──────────────────┘


In [52]:
# 전월 대비 Cart 이탈률 변화(%p) 계산

duckdb.sql(f"""
    WITH monthly_cart AS (
        SELECT
            STRFTIME(first_cart_after_view, '%Y-%m') AS month,

            ROUND(
                SUM(CASE WHEN converted = 0
                         THEN 1 ELSE 0 END) * 100.0
                / COUNT(*),
                2
            ) AS abandonment_rate

        FROM read_parquet('{cart_abandonment_parquet}')

        GROUP BY month
    )

    SELECT
        month,
        abandonment_rate,

        ROUND(
            abandonment_rate
            - LAG(abandonment_rate) OVER (ORDER BY month),
            2
        ) AS abandonment_change_pp

    FROM monthly_cart

    ORDER BY month
""").show()

┌─────────┬──────────────────┬───────────────────────┐
│  month  │ abandonment_rate │ abandonment_change_pp │
│ varchar │      double      │        double         │
├─────────┼──────────────────┼───────────────────────┤
│ 2019-12 │            53.61 │                  NULL │
│ 2020-01 │            51.95 │                 -1.66 │
│ 2020-02 │            50.67 │                 -1.28 │
│ 2020-03 │            49.84 │                 -0.83 │
│ 2020-04 │             55.2 │                  5.36 │
└─────────┴──────────────────┴───────────────────────┘



In [54]:
# Cart 발생 시간대별 구매 완료율과 이탈률 계산

duckdb.sql(f"""
    SELECT
        EXTRACT(HOUR FROM first_cart_after_view) AS cart_hour,

        COUNT(*) AS carted_count,

        SUM(
            CASE WHEN converted = 1
            THEN 1 ELSE 0 END
        ) AS converted_count,

        SUM(
            CASE WHEN converted = 0
            THEN 1 ELSE 0 END
        ) AS abandoned_count,

        ROUND(
            SUM(CASE WHEN converted = 1
                     THEN 1 ELSE 0 END) * 100.0
            / COUNT(*),
            2
        ) AS conversion_rate,

        ROUND(
            SUM(CASE WHEN converted = 0
                     THEN 1 ELSE 0 END) * 100.0
            / COUNT(*),
            2
        ) AS abandonment_rate

    FROM read_parquet('{cart_abandonment_parquet}')

    GROUP BY cart_hour
    ORDER BY cart_hour
""").show()

┌───────────┬──────────────┬─────────────────┬─────────────────┬─────────────────┬──────────────────┐
│ cart_hour │ carted_count │ converted_count │ abandoned_count │ conversion_rate │ abandonment_rate │
│   int64   │    int64     │     int128      │     int128      │     double      │      double      │
├───────────┼──────────────┼─────────────────┼─────────────────┼─────────────────┼──────────────────┤
│         0 │        46260 │           20022 │           26238 │           43.28 │            56.72 │
│         1 │        74596 │           30449 │           44147 │           40.82 │            59.18 │
│         2 │       155009 │           67168 │           87841 │           43.33 │            56.67 │
│         3 │       301633 │          144942 │          156691 │           48.05 │            51.95 │
│         4 │       441911 │          217435 │          224476 │            49.2 │             50.8 │
│         5 │       538764 │          264855 │          273909 │           49.16 │

In [55]:
# Cart 이탈률이 높은 시간대 순으로 확인

duckdb.sql(f"""
    SELECT
        EXTRACT(HOUR FROM first_cart_after_view) AS cart_hour,

        COUNT(*) AS carted_count,

        ROUND(
            SUM(CASE WHEN converted = 0
                     THEN 1 ELSE 0 END) * 100.0
            / COUNT(*),
            2
        ) AS abandonment_rate

    FROM read_parquet('{cart_abandonment_parquet}')

    GROUP BY cart_hour
    ORDER BY abandonment_rate DESC
""").show()

┌───────────┬──────────────┬──────────────────┐
│ cart_hour │ carted_count │ abandonment_rate │
│   int64   │    int64     │      double      │
├───────────┼──────────────┼──────────────────┤
│         1 │        74596 │            59.18 │
│         0 │        46260 │            56.72 │
│         2 │       155009 │            56.67 │
│        17 │       424560 │            56.35 │
│        16 │       473703 │            55.95 │
│        18 │       350791 │             55.3 │
│        15 │       501180 │            55.26 │
│        14 │       503833 │            53.65 │
│        19 │       243593 │            53.55 │
│        23 │        44545 │            53.48 │
│         · │          ·   │              ·   │
│         · │          ·   │              ·   │
│         · │          ·   │              ·   │
│        21 │        92217 │            51.62 │
│        11 │       581070 │            51.23 │
│        22 │        60684 │            51.21 │
│         6 │       604296 │            

In [58]:
# Cart 발생 시간을 4개 시간대 그룹으로 나누어 이탈률 비교

duckdb.sql(f"""
    WITH cart_time_group AS (
        SELECT
            *,
            CASE
                WHEN EXTRACT(HOUR FROM first_cart_after_view) BETWEEN 0 AND 5
                    THEN '새벽'
                WHEN EXTRACT(HOUR FROM first_cart_after_view) BETWEEN 6 AND 11
                    THEN '오전'
                WHEN EXTRACT(HOUR FROM first_cart_after_view) BETWEEN 12 AND 17
                    THEN '오후'
                ELSE '저녁'
            END AS time_group

        FROM read_parquet('{cart_abandonment_parquet}')
    )

    SELECT
        time_group,

        COUNT(*) AS carted_count,

        SUM(
            CASE WHEN converted = 1
            THEN 1 ELSE 0 END
        ) AS converted_count,

        SUM(
            CASE WHEN converted = 0
            THEN 1 ELSE 0 END
        ) AS abandoned_count,

        ROUND(
            SUM(CASE WHEN converted = 1
                     THEN 1 ELSE 0 END) * 100.0
            / COUNT(*),
            2
        ) AS conversion_rate,

        ROUND(
            SUM(CASE WHEN converted = 0
                     THEN 1 ELSE 0 END) * 100.0
            / COUNT(*),
            2
        ) AS abandonment_rate

    FROM cart_time_group

    GROUP BY time_group

    ORDER BY
        CASE time_group
            WHEN '새벽' THEN 1
            WHEN '오전' THEN 2
            WHEN '오후' THEN 3
            WHEN '저녁' THEN 4
        END
""").show()

┌────────────┬──────────────┬─────────────────┬─────────────────┬─────────────────┬──────────────────┐
│ time_group │ carted_count │ converted_count │ abandoned_count │ conversion_rate │ abandonment_rate │
│  varchar   │    int64     │     int128      │     int128      │     double      │      double      │
├────────────┼──────────────┼─────────────────┼─────────────────┼─────────────────┼──────────────────┤
│ 새벽       │      1558173 │          744871 │          813302 │            47.8 │             52.2 │
│ 오전       │      3742868 │         1841097 │         1901771 │           49.19 │            50.81 │
│ 오후       │      2955869 │         1356975 │         1598894 │           45.91 │            54.09 │
│ 저녁       │       943489 │          436495 │          506994 │           46.26 │            53.74 │
└────────────┴──────────────┴─────────────────┴─────────────────┴─────────────────┴──────────────────┘



In [60]:
# 구매 완료된 세션×상품의 Cart → Purchase 소요시간 계산

duckdb.sql(f"""
    SELECT
        COUNT(*) AS converted_count,

        ROUND(
            AVG(
                DATE_DIFF(
                    'second',
                    first_cart_after_view,
                    first_purchase_after_cart
                )
            ) / 60.0,
            2
        ) AS avg_minutes,

        ROUND(
            MEDIAN(
                DATE_DIFF(
                    'second',
                    first_cart_after_view,
                    first_purchase_after_cart
                )
            ) / 60.0,
            2
        ) AS median_minutes,

        ROUND(
            QUANTILE_CONT(
                DATE_DIFF(
                    'second',
                    first_cart_after_view,
                    first_purchase_after_cart
                ),
                0.75
            ) / 60.0,
            2
        ) AS p75_minutes,

        ROUND(
            QUANTILE_CONT(
                DATE_DIFF(
                    'second',
                    first_cart_after_view,
                    first_purchase_after_cart
                ),
                0.90
            ) / 60.0,
            2
        ) AS p90_minutes

    FROM read_parquet('{cart_abandonment_parquet}')

    WHERE converted = 1
""").show()

┌─────────────────┬─────────────┬────────────────┬─────────────┬─────────────┐
│ converted_count │ avg_minutes │ median_minutes │ p75_minutes │ p90_minutes │
│      int64      │   double    │     double     │   double    │   double    │
├─────────────────┼─────────────┼────────────────┼─────────────┼─────────────┤
│         4379438 │        2.25 │           1.33 │        2.67 │        4.78 │
└─────────────────┴─────────────┴────────────────┴─────────────┴─────────────┘



In [62]:
# Cart → Purchase 소요시간을 구간별로 나누어 분포 확인

duckdb.sql(f"""
    WITH purchase_time AS (
        SELECT
            DATE_DIFF(
                'second',
                first_cart_after_view,
                first_purchase_after_cart
            ) / 60.0 AS minutes_to_purchase

        FROM read_parquet('{cart_abandonment_parquet}')

        WHERE converted = 1
    )

    SELECT
        CASE
            WHEN minutes_to_purchase < 1
                THEN '1분 미만'

            WHEN minutes_to_purchase < 5
                THEN '1~5분'

            WHEN minutes_to_purchase < 10
                THEN '5~10분'

            WHEN minutes_to_purchase < 30
                THEN '10~30분'

            WHEN minutes_to_purchase < 60
                THEN '30~60분'

            ELSE '60분 이상'
        END AS purchase_time_group,

        COUNT(*) AS purchase_count,

        ROUND(
            COUNT(*) * 100.0
            / SUM(COUNT(*)) OVER (),
            2
        ) AS purchase_share

    FROM purchase_time

    GROUP BY purchase_time_group

    ORDER BY
        CASE purchase_time_group
            WHEN '1분 미만' THEN 1
            WHEN '1~5분' THEN 2
            WHEN '5~10분' THEN 3
            WHEN '10~30분' THEN 4
            WHEN '30~60분' THEN 5
            ELSE 6
        END
""").show()

┌─────────────────────┬────────────────┬────────────────┐
│ purchase_time_group │ purchase_count │ purchase_share │
│       varchar       │     int64      │     double     │
├─────────────────────┼────────────────┼────────────────┤
│ 1분 미만            │        1723297 │          39.35 │
│ 1~5분               │        2249603 │          51.37 │
│ 5~10분              │         298811 │           6.82 │
│ 10~30분             │          99116 │           2.26 │
│ 30~60분             │           7973 │           0.18 │
│ 60분 이상           │            638 │           0.01 │
└─────────────────────┴────────────────┴────────────────┘



In [63]:
# Cart → Purchase 소요시간의 상위 분위수와 최대값 확인

duckdb.sql(f"""
    WITH purchase_time AS (
        SELECT
            DATE_DIFF(
                'second',
                first_cart_after_view,
                first_purchase_after_cart
            ) / 60.0 AS minutes_to_purchase

        FROM read_parquet('{cart_abandonment_parquet}')

        WHERE converted = 1
    )

    SELECT
        ROUND(QUANTILE_CONT(minutes_to_purchase, 0.90), 2) AS p90_minutes,
        ROUND(QUANTILE_CONT(minutes_to_purchase, 0.95), 2) AS p95_minutes,
        ROUND(QUANTILE_CONT(minutes_to_purchase, 0.99), 2) AS p99_minutes,
        ROUND(QUANTILE_CONT(minutes_to_purchase, 0.999), 2) AS p999_minutes,
        ROUND(MAX(minutes_to_purchase), 2) AS max_minutes

    FROM purchase_time
""").show()

┌─────────────┬─────────────┬─────────────┬──────────────┬─────────────┐
│ p90_minutes │ p95_minutes │ p99_minutes │ p999_minutes │ max_minutes │
│   double    │   double    │   double    │    double    │   double    │
├─────────────┼─────────────┼─────────────┼──────────────┼─────────────┤
│        4.78 │        6.93 │       15.78 │        36.18 │      299.45 │
└─────────────┴─────────────┴─────────────┴──────────────┴─────────────┘



In [64]:
# 24시간을 초과한 Purchase 연결이 남아 있는지 확인

duckdb.sql(f"""
    SELECT
        COUNT(*) AS over_24h_count

    FROM read_parquet('{cart_abandonment_parquet}')

    WHERE converted = 1
      AND DATE_DIFF(
            'second',
            first_cart_after_view,
            first_purchase_after_cart
          ) >= 86400
""").show()

┌────────────────┐
│ over_24h_count │
│     int64      │
├────────────────┤
│              0 │
└────────────────┘



In [66]:
#-------------------------------------------------------------------------------------------------------------------------#

In [68]:
# [Tableau용 저장 파일]
# 월별 Cart 이탈률 결과를 CSV로 저장
monthly_cart_path = r"../data/marts/dashboard_monthly_cart_abandonment.csv"

Path(monthly_cart_path).unlink(missing_ok=True)

duckdb.sql(f"""
    COPY (
        SELECT
            STRFTIME(first_cart_after_view, '%Y-%m') AS month,
            COUNT(*) AS carted_count,

            SUM(CASE WHEN converted = 1
                     THEN 1 ELSE 0 END) AS converted_count,

            SUM(CASE WHEN converted = 0
                     THEN 1 ELSE 0 END) AS abandoned_count,

            ROUND(
                SUM(CASE WHEN converted = 1
                         THEN 1 ELSE 0 END) * 100.0
                / COUNT(*),
                2
            ) AS conversion_rate,

            ROUND(
                SUM(CASE WHEN converted = 0
                         THEN 1 ELSE 0 END) * 100.0
                / COUNT(*),
                2
            ) AS abandonment_rate

        FROM read_parquet('{cart_abandonment_parquet}')

        GROUP BY month
        ORDER BY month
    )
    TO '{monthly_cart_path}'
    (HEADER, DELIMITER ',')
""")

In [72]:
# [Tableau용 저장 파일]
# 시간대 그룹별 Cart 이탈률 저장

cart_time_path = r"../data/marts/dashboard_cart_time.csv"

Path(cart_time_path).unlink(missing_ok=True)

duckdb.sql(f"""
    COPY (
        WITH cart_time_group AS (
            SELECT
                *,
                CASE
                    WHEN EXTRACT(HOUR FROM first_cart_after_view) BETWEEN 0 AND 5
                        THEN '새벽'
                    WHEN EXTRACT(HOUR FROM first_cart_after_view) BETWEEN 6 AND 11
                        THEN '오전'
                    WHEN EXTRACT(HOUR FROM first_cart_after_view) BETWEEN 12 AND 17
                        THEN '오후'
                    ELSE '저녁'
                END AS time_group

            FROM read_parquet('{cart_abandonment_parquet}')
        )

        SELECT
            time_group,
            COUNT(*) AS carted_count,

            SUM(CASE WHEN converted = 1
                     THEN 1 ELSE 0 END) AS converted_count,

            SUM(CASE WHEN converted = 0
                     THEN 1 ELSE 0 END) AS abandoned_count,

            ROUND(
                SUM(CASE WHEN converted = 0
                         THEN 1 ELSE 0 END) * 100.0
                / COUNT(*),
                2
            ) AS abandonment_rate

        FROM cart_time_group

        GROUP BY time_group
    )
    TO '{cart_time_path}'
    (HEADER, DELIMITER ',')
""")

In [75]:
# [검증용 코드]
# Tableau에 연결한 월별 Cart 이탈률 CSV의 실제 값 확인

duckdb.sql("""
    SELECT *
    FROM read_csv_auto(
        '../data/marts/dashboard_monthly_cart_abandonment.csv'
    )
    ORDER BY month
""").show()

┌─────────┬──────────────┬─────────────────┬─────────────────┬─────────────────┬──────────────────┐
│  month  │ carted_count │ converted_count │ abandoned_count │ conversion_rate │ abandonment_rate │
│ varchar │    int64     │      int64      │      int64      │     double      │      double      │
├─────────┼──────────────┼─────────────────┼─────────────────┼─────────────────┼──────────────────┤
│ 2019-12 │      2298442 │         1066190 │         1232252 │           46.39 │            53.61 │
│ 2020-01 │      1522267 │          731469 │          790798 │           48.05 │            51.95 │
│ 2020-02 │      1585664 │          782129 │          803535 │           49.33 │            50.67 │
│ 2020-03 │      1861838 │          933979 │          927859 │           50.16 │            49.84 │
│ 2020-04 │      1932188 │          865671 │         1066517 │            44.8 │             55.2 │
└─────────┴──────────────┴─────────────────┴─────────────────┴─────────────────┴──────────────────┘


In [77]:
# [검증용 코드]
# View 발생 월 기준과 Cart 발생 월 기준의 이탈률을 나란히 비교

duckdb.sql(f"""
    WITH by_view_month AS (
        SELECT
            STRFTIME(first_view_time, '%Y-%m') AS month,
            COUNT(*) AS carted_count_view_month,

            ROUND(
                SUM(CASE WHEN converted = 0 THEN 1 ELSE 0 END)
                * 100.0 / COUNT(*),
                2
            ) AS abandonment_rate_view_month

        FROM read_parquet('{cart_abandonment_parquet}')

        GROUP BY month
    ),

    by_cart_month AS (
        SELECT
            STRFTIME(first_cart_after_view, '%Y-%m') AS month,
            COUNT(*) AS carted_count_cart_month,

            ROUND(
                SUM(CASE WHEN converted = 0 THEN 1 ELSE 0 END)
                * 100.0 / COUNT(*),
                2
            ) AS abandonment_rate_cart_month

        FROM read_parquet('{cart_abandonment_parquet}')

        GROUP BY month
    )

    SELECT
        v.month,
        v.carted_count_view_month,
        v.abandonment_rate_view_month,
        c.carted_count_cart_month,
        c.abandonment_rate_cart_month

    FROM by_view_month v

    JOIN by_cart_month c
        ON v.month = c.month

    ORDER BY v.month
""").show()

┌─────────┬─────────────────────────┬─────────────────────────────┬─────────────────────────┬─────────────────────────────┐
│  month  │ carted_count_view_month │ abandonment_rate_view_month │ carted_count_cart_month │ abandonment_rate_cart_month │
│ varchar │          int64          │           double            │          int64          │           double            │
├─────────┼─────────────────────────┼─────────────────────────────┼─────────────────────────┼─────────────────────────────┤
│ 2019-12 │                 2298442 │                       53.61 │                 2298442 │                       53.61 │
│ 2020-01 │                 1522277 │                       51.95 │                 1522267 │                       51.95 │
│ 2020-02 │                 1585659 │                       50.67 │                 1585664 │                       50.67 │
│ 2020-03 │                 1861838 │                       49.84 │                 1861838 │                       49.84 │
│ 2020-0

In [101]:
#-------------------------------------------------------------------------------------------------------------------------#